# Requirements Installation


In [ ]:
!pip install transformers datasets accelerate -q
!pip install huggingface_hub

In [ ]:
!pip install bertopic -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 6.2 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import re

# Helper functions declaration


In [ ]:
def clean_date(d):
  d = str(d).strip()
  if d.startswith('http') or d == '' or len(d) > 20 or d == 'nan':
    return np.nan
  try:
      return pd.to_datetime(d).strftime('%Y-%m-%d')
  except:
      return np.nan

In [ ]:
def clean_text(text):
  text = str(text)
  text = re.sub(r'^[A-Z][a-z]+ [A-Z][a-z]+ says\s*', '', text)
  text = re.sub(r'Featured image.*$', '', text, flags=re.I)
  text = re.sub(r'Image source.*$', '', text, flags=re.I)
  text = re.sub(r'\s+', ' ', text).strip()
  return text

# Data Preprocessing

In [ ]:
df_fake = pd.read_csv('Fake.csv', engine='python', on_bad_lines='warn', quotechar='"')

print(len(df_fake))
df_fake = df_fake.drop_duplicates(subset=['title', 'text']).reset_index(drop=True)
print(len(df_fake))

23481
17908


In [ ]:
df_fake['date'] = df_fake['date'].apply(clean_date)
df_fake = df_fake.dropna(subset=['date']).reset_index(drop=True)

df_fake['text_clean'] = df_fake['text'].apply(clean_text)
df_fake['title_clean'] = df_fake['title'].apply(clean_text)

df_fake['content'] = df_fake['title_clean'] + " " + df_fake['text_clean']
df_fake['label'] = 0

# Model Training

In [ ]:
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer



In [ ]:
topic_model_fake = BERTopic(language="english",
                            calculate_probabilities=True,
                            verbose=True)

texts_fake = df_fake['content'].tolist()

topics_fake, probs_fake = topic_model_fake.fit_transform(texts_fake)

2025-12-14 10:27:29,541 - BERTopic - Embedding - Transforming documents to embeddings.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (4 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/560 [00:00<?, ?it/s]

2025-12-14 11:00:56,182 - BERTopic - Embedding - Completed ✓
2025-12-14 11:00:56,184 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-12-14 11:01:40,394 - BERTopic - Dimensionality - Completed ✓
2025-12-14 11:01:40,397 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-12-14 11:02:41,788 - BERTopic - Cluster - Completed ✓
2025-12-14 11:02:41,801 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-12-14 11:02:50,690 - BERTopic - Representation - Completed ✓


In [ ]:
model_dir = "bertopic_fake_news_detector"
topic_model_fake.save(model_dir, save_embedding_model=True, save_ctfidf=True,
                      serialization="pytorch")

from huggingface_hub import upload_folder

if hasattr(topic_model_fake.ctfidf_model, '_idf_diag'):
    import scipy.sparse as sp
    idf_diag = topic_model_fake.ctfidf_model._idf_diag

    if sp.issparse(idf_diag):
        idf_values = idf_diag.diagonal()
    else:
        idf_values = idf_diag

    topic_model_fake.ctfidf_model._idf_diag = idf_values

upload_folder(
    folder_path=model_dir,
    repo_id="evstiars/bertopic-fake-news-detector",
    commit_message="Full BERTopic model upload with all files",
)

print("Загрузка завершена! Теперь BERTopic.load сработает.")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ctor/topic_embeddings.bin: 100%|##########|  416kB /  416kB            

  ..._news_detector/ctfidf.bin: 100%|##########| 12.7MB / 12.7MB            

No files have been modified since last commit. Skipping to prevent empty commit.


Загрузка завершена! Теперь BERTopic.load сработает.
